In [0]:
# ============================================================
# 03_GOLD_FEATURES
# Silver → Feature Engineering → Gold Delta Table
# ============================================================

SILVER_TABLE = "workspace.default.silver_loan_applications"
GOLD_TABLE = "workspace.default.gold_loan_features"

df = spark.table(SILVER_TABLE)
print("✅ Rows from Silver:", df.count())
print("✅ Columns:", len(df.columns))

✅ Rows from Silver: 307511
✅ Columns: 75


In [0]:
from pyspark.sql.functions import col, round as spark_round, when

df = df \
    .withColumn("CREDIT_INCOME_RATIO",
        spark_round(col("AMT_CREDIT") / (col("AMT_INCOME_TOTAL") + 1), 4)) \
    .withColumn("ANNUITY_INCOME_RATIO",
        spark_round(col("AMT_ANNUITY") / (col("AMT_INCOME_TOTAL") + 1), 4)) \
    .withColumn("CREDIT_GOODS_RATIO",
        spark_round(col("AMT_CREDIT") / (col("AMT_GOODS_PRICE") + 1), 4)) \
    .withColumn("INCOME_PER_PERSON",
        spark_round(col("AMT_INCOME_TOTAL") / (col("CNT_FAM_MEMBERS") + 1), 4))

print("✅ Financial ratio features added")

✅ Financial ratio features added


In [0]:
# DAYS_BIRTH and DAYS_EMPLOYED are negative numbers in this dataset
df = df \
    .withColumn("AGE_YEARS",
        spark_round((-col("DAYS_BIRTH")) / 365, 1)) \
    .withColumn("EMPLOYMENT_YEARS",
        spark_round(
            when(col("DAYS_EMPLOYED") > 0, 0)  # anomaly: positive = unemployed
            .otherwise(-col("DAYS_EMPLOYED") / 365), 1)) \
    .withColumn("IS_UNEMPLOYED",
        when(col("DAYS_EMPLOYED") > 0, 1).otherwise(0)) \
    .withColumn("EMPLOYMENT_TO_AGE_RATIO",
        spark_round(
            when(col("DAYS_EMPLOYED") > 0, 0)
            .otherwise(-col("DAYS_EMPLOYED") / (-col("DAYS_BIRTH") + 1)), 4))

print("✅ Age and employment features added")

✅ Age and employment features added


In [0]:
# External source average (credit score proxy)
df = df \
    .withColumn("EXT_SOURCE_MEAN",
        spark_round((col("EXT_SOURCE_2") + col("EXT_SOURCE_3")) / 2, 4)) \
    .withColumn("EXT_SOURCE_MIN",
        when(col("EXT_SOURCE_2") < col("EXT_SOURCE_3"),
             col("EXT_SOURCE_2")).otherwise(col("EXT_SOURCE_3")))

# Total documents submitted
doc_cols = [c for c in df.columns if c.startswith("FLAG_DOCUMENT_")]
from pyspark.sql.functions import lit
df = df.withColumn("TOTAL_DOCS_SUBMITTED",
    sum([col(c) for c in doc_cols]))

# Social circle risk
df = df \
    .withColumn("SOCIAL_CIRCLE_DEFAULT_RATE",
        spark_round(
            (col("DEF_30_CNT_SOCIAL_CIRCLE") + col("DEF_60_CNT_SOCIAL_CIRCLE")) /
            (col("OBS_30_CNT_SOCIAL_CIRCLE") + col("OBS_60_CNT_SOCIAL_CIRCLE") + 1), 4))

print("✅ Risk and document features added")

✅ Risk and document features added


In [0]:
# Create a human-readable risk segment for analytics
df = df.withColumn("RISK_SEGMENT",
    when(col("EXT_SOURCE_MEAN") >= 0.6, "LOW_RISK")
    .when(col("EXT_SOURCE_MEAN") >= 0.4, "MEDIUM_RISK")
    .when(col("EXT_SOURCE_MEAN") >= 0.2, "HIGH_RISK")
    .otherwise("VERY_HIGH_RISK"))

print("✅ Risk segment labels added")
df.groupBy("RISK_SEGMENT").count().orderBy("RISK_SEGMENT").show()

✅ Risk segment labels added
+--------------+------+
|  RISK_SEGMENT| count|
+--------------+------+
|     HIGH_RISK| 89053|
|      LOW_RISK| 77254|
|   MEDIUM_RISK|116955|
|VERY_HIGH_RISK| 24249|
+--------------+------+



In [0]:
from pyspark.sql.functions import current_timestamp, lit

df_gold = df \
    .withColumn("_gold_processed_at", current_timestamp()) \
    .withColumn("_layer", lit("gold"))

df_gold.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(GOLD_TABLE)

print(f"✅ Gold table written: {GOLD_TABLE}")

✅ Gold table written: workspace.default.gold_loan_features


In [0]:
from pyspark.sql.functions import sum as spark_sum, count, col, round as spark_round

df_check = spark.table(GOLD_TABLE)
print("✅ Gold rows:", df_check.count())
print("✅ Gold columns:", len(df_check.columns))

# Key business insight: Default rate by risk segment
print("\n📊 Default Rate by Risk Segment:")
df_check.groupBy("RISK_SEGMENT") \
    .agg(
        spark_round(
            (spark_sum(col("TARGET")) / count("TARGET") * 100), 2
        ).alias("DEFAULT_RATE_%"),
        count("TARGET").alias("TOTAL_APPLICANTS")
    ).orderBy("DEFAULT_RATE_%", ascending=False).show()

✅ Gold rows: 307511
✅ Gold columns: 89

📊 Default Rate by Risk Segment:
+--------------+--------------+----------------+
|  RISK_SEGMENT|DEFAULT_RATE_%|TOTAL_APPLICANTS|
+--------------+--------------+----------------+
|VERY_HIGH_RISK|         19.91|           24249|
|     HIGH_RISK|          11.4|           89053|
|   MEDIUM_RISK|          6.59|          116955|
|      LOW_RISK|          2.78|           77254|
+--------------+--------------+----------------+



In [0]:
# Serverless doesn't support ZORDER config override
# OPTIMIZE without ZORDER works fine

spark.sql(f"OPTIMIZE {GOLD_TABLE}")
print("✅ OPTIMIZE applied")

# Time travel proof
spark.sql(f"DESCRIBE HISTORY {GOLD_TABLE}").show(5, truncate=False)

✅ OPTIMIZE applied
+-------+-------------------+--------------+-----------------------+---------------------------------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+----+------------------+------------------------------------+------------------------+-----------+-----------------+-------------+------------------------------------------------------------------------------------------------------------------------------------------------+------------+--------------------------------------------------+
|version|timestamp          |userId        |userName               |operation                        |operationParameters                                                                      